# Demo Portfolio Construction and Backtest 

Code by obed Mireku

This file will include the codes for backtesting a trading strategy using technical signals. I will use the two signals developed in *Demo Quantitative Stock Selection Model.ipynb* in this demo. The signals have been stored in the files "classdata/Signal1.csv" and "classdata/Signal2.csv".

In [1]:
import pandas as pd
import numpy as np
df1 = pd.read_csv('classdata/Signal1.csv')
df1['datadate']=pd.to_datetime(df1['datadate'],format="%Y-%m-%d")
df1.sort_values(by=['LPERMNO', 'datadate'], inplace=True)
df1.reset_index(drop=True, inplace=True)
df1.head()

,LPERMNO,datadate,Date,Signal
0,10421,2000-03-31,200009,1.004885
1,10421,2000-03-31,200012,1.004885
2,10421,2000-03-31,200103,1.004885
3,10421,2000-03-31,200106,1.004885
4,10421,2001-03-31,200109,0.772176


In [2]:
df2 = pd.read_csv('classdata/Signal2.csv')
df2['STATPERS']=pd.to_datetime(df2['STATPERS'],format="%Y-%m-%d")
df2.sort_values(by=['PERMNO', 'STATPERS'], inplace=True)
df2.reset_index(drop=True, inplace=True)
df2.head()

,PERMNO,STATPERS,Date,Signal
0,10421,2013-03-18,201303,0.000160
1,10421,2013-06-18,201306,0.000101
2,10421,2013-09-18,201309,0.000065
3,10421,2013-12-18,201312,0.000033
4,10421,2014-03-18,201403,0.000213


In [3]:
df3 = pd.read_csv('classdata/return.csv')
df3.sort_values(by=['permno', 'yyyymm'], inplace=True)
df3.reset_index(drop=True, inplace=True)
df3.head()

,permno,yyyymm,prc,fret1
0,10421,200003,292.8271,-0.031083
1,10421,200006,338.7134,-0.183489
2,10421,200009,304.2295,0.041123
3,10421,200012,261.6243,0.177023
4,10421,200103,287.4984,0.319456


## Generate Keys and Merge Data Frames for Signal 1

We generate a column of keys which is "PERMONDate", for example, 10002200606. Since, PERMNO and Date are integers in all dataframes, we can just let key = PERMON*100000+Date.

In [4]:
df1["key"]=df1["LPERMNO"]*1000000+df1["Date"]
df2["key"]=df2["PERMNO"]*1000000+df2["Date"]
df3["key"]=df3["permno"]*1000000+df3["yyyymm"]

Then we merge data frames using keys into one,  drop NaNs, and sort the new data frame. We will only use Signal 1 in backtesting first. 

In [5]:
df=pd.merge(df1[["key","LPERMNO","Date","Signal"]],df3[["key","prc","fret1"]],on="key")
df.dropna(how="any",inplace=True)
df.sort_values(by=['LPERMNO', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,key,LPERMNO,Date,Signal,prc,fret1
0,10421200009,10421,200009,1.004885,304.2295,0.041123
1,10421200012,10421,200012,1.004885,261.6243,0.177023
2,10421200103,10421,200103,1.004885,287.4984,0.319456
3,10421200106,10421,200106,1.004885,292.2451,-0.115506
4,10421200109,10421,200109,0.772176,332.6072,0.139668


## Backtesting for Signal 1

We then implement the trading strategy and generat the returns of the equally weighted long-short portfolio.

In [6]:
import numpy as np
dfbenchmark=pd.read_csv('classdata/benchmark.csv')

benchmark=[]
quarterlist=[]
quarterlyreturn=[]
longreturn=[]
shortreturn=[]
for quarter in np.sort(df.Date.unique()):
    dftemp = df[df.Date==quarter].copy()   #Get data for the same quarter and save it in dftemp
    if len(dftemp)>=200:                   #Only continue if there are more than 200 stocks. 
        quarterlist.append(quarter)        #Get quarter ID
        benchmark.append(dfbenchmark[dfbenchmark.yyyymm==quarter].benchmark.iloc[0])#Get benchmark
        dftemp.sort_values(by='Signal', ascending=True, inplace=True)  #Sort by Signal 1
        #Below, we compute the returns of equally weighted long-short portfolio.
        longreturn.append(dftemp["fret1"][0:100].mean())
        shortreturn.append(-dftemp["fret1"][-100:].mean())        
        quarterlyreturn.append(dftemp["fret1"][0:100].mean()-dftemp["fret1"][-100:].mean())

#Collect the results and put them into a data frame
dfresult=pd.DataFrame(zip(quarterlist,longreturn,shortreturn,quarterlyreturn,benchmark),
                        columns=["Date","LongReturn","ShortReturn","QuarterlyReturn","Benchmark"])
dfresult.head()

,Date,LongReturn,ShortReturn,QuarterlyReturn,Benchmark
0,200106,0.002916,0.022184,0.025100,0.049142
1,200109,0.010131,0.005297,0.015428,0.000834
2,200112,0.018471,-0.001254,0.017217,0.090308
3,200203,0.010068,0.027269,0.037337,-0.061604
4,200206,0.029019,0.049327,0.078346,-0.000755


Finally, we generate the performance metrics.

In [7]:
dfresult["ExcessReturn"]=dfresult.QuarterlyReturn

In [8]:
import math
#Average ExcessReturn
print(dfresult.ExcessReturn.mean())
#t-statistics 
print(dfresult.ExcessReturn.mean()/dfresult.ExcessReturn.std()*math.sqrt(len(dfresult)))

0.02494350974358974
6.118540270718748


In [9]:
#Informatin ratio
print(dfresult.ExcessReturn.mean()/dfresult.ExcessReturn.std())

0.6927882631023695


## Generate Keys and Merge Data Frames for Signal 2

In [10]:
df=pd.merge(df2[["key","PERMNO","Date","Signal"]],df3[["key","prc","fret1"]],on="key")
df.dropna(how="any",inplace=True)
df.sort_values(by=['PERMNO', 'Date'], inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

,key,PERMNO,Date,Signal,prc,fret1
0,10421201303,10421,201303,0.000160,298.2019,0.174711
1,10421201306,10421,201306,0.000101,304.0173,-0.360624
2,10421201309,10421,201309,0.000065,313.8364,0.235842
3,10421201312,10421,201312,0.000033,301.2982,-0.066317
4,10421201403,10421,201403,0.000213,304.9911,-0.079078


## Backtesting for Signal 2

Almost similar to the procedure done with Signal 1. 

In [11]:
benchmark=[]
quarterlist=[]
quarterlyreturn=[]
longreturn=[]
shortreturn=[]
for quarter in np.sort(df.Date.unique()):
    dftemp = df[df.Date==quarter].copy()   #Get data for the same quarter and save it in dftemp
    if len(dftemp)>=200:                   #Only continue if there are more than 200 stocks. 
        quarterlist.append(quarter)        #Get quarter ID
        benchmark.append(dfbenchmark[dfbenchmark.yyyymm==quarter].benchmark.iloc[0])#Get benchmark
        dftemp.sort_values(by='Signal', ascending=True, inplace=True)  #Sort by Signal 2
        #Below, we compute the returns of equally weighted long-short portfolio.
        longreturn.append(dftemp["fret1"][0:100].mean())
        shortreturn.append(-dftemp["fret1"][-100:].mean())        
        quarterlyreturn.append(dftemp["fret1"][0:100].mean()-dftemp["fret1"][-100:].mean())

#Collect the results and put them into a data frame
dfresult=pd.DataFrame(zip(quarterlist,longreturn,shortreturn,quarterlyreturn,benchmark),
                        columns=["Date","LongReturn","ShortReturn","QuarterlyReturn","Benchmark"])
dfresult.head()

,Date,LongReturn,ShortReturn,QuarterlyReturn,Benchmark
0,201303,0.006579,0.007692,0.014271,0.072923
1,201306,-0.003889,0.016634,0.012745,0.080640
2,201309,-0.001869,-0.007755,-0.009624,-0.007277
3,201312,-0.025012,-0.010426,-0.035438,-0.054351
4,201403,0.006740,-0.023559,-0.016819,0.042303


In [12]:
dfresult["ExcessReturn"]=dfresult.QuarterlyReturn

In [13]:
import math
#Average ExcessReturn
print(dfresult.ExcessReturn.mean())
#t-statistics 
print(dfresult.ExcessReturn.mean()/dfresult.ExcessReturn.std()*math.sqrt(len(dfresult)))

-0.0031707762500000013
-0.761738199060178


In [14]:
#Informatin ratio
print(dfresult.ExcessReturn.mean()/dfresult.ExcessReturn.std())

-0.1904345497650445
